In [59]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
from global_land_mask import globe

In [ ]:
ens = xr.open_dataset('ensemble_data_stereo_coords.nc')

In [48]:
ice_dum = xr.open_dataset('ice_data.nc')

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':ccrs.NorthPolarStereo})
ax.coastlines()
ax.plot()

In [32]:
# making sure that ice data has the same time index as the ensemble data
ice_dum = xr.open_dataset('ice_data.nc')
ice_dum = ice_dum.sel(time=slice('2023-05-08','2023-06-13'))

start = ice_dum.time.values[0]
end = ice_dum.time.values[-1] + pd.Timedelta(hours=23)   # include all 24 hours of the last day

hourly_times = pd.date_range(start, end, freq="1h")

ice = ice_dum.reindex(time=hourly_times, method="ffill")

ice = ice.drop_sel(time='2023-05-24 01:00:00') # missing from ens dataset

ice = ice.rename({'time':'launch_time'})

# Masking Functions

In [63]:
"""
Ice masking functions
"""

def closest_icecoord(value, search_array):   
    """
    Find the closest value in a given array (search_array) to a single value given by the user (x).
    For use in conjunction with apply_ufunc.
    Inputs:
    - values: the array of values that you want to 
    - search_array: the array that contains the values that you want to find the closest value of.
    Outputs:
    - an array 
    """
    value_dif = np.abs(value - search_array)
    closest_value = search_array[np.argmin(value_dif)]
    return closest_value
    
    return search_array[np.argmin(value_dif, axis=-1)]

def determine_ice_coord(closest_x, closest_y, ice_data):
    """
    Determine the ice coodrdinate at that location from the ice data.
    Inputs:
    - closest_x, closest_y: the closest x and y coordinates held in the ice data netCDF file.
    Outputs:
    - z: the ice coordinate that corresponds to these coordinates.
    """
    z = ice_data['z'].sel(x=closest_x, y=closest_y).values
    return z

def ice_threshold(z, thresh):
    """
    Determing if an ice coordinate is above or below a given threshshold, where the threshhold represents a percentage from 0 to 100.
    Inputs:
    - z: the ice value you want to determine if it is above or below a given value
    - thresh: the user-defined percentage threshhold.
    Outputs:
    - 1 if the z value is determined to be ice, and 0 if it is not ice.
    """
    return (z >= thresh).astype(int)

def ice_interpolation(x,y):
    """
    Finding the squared distance of an x-y coordinate in the stereographic projection from the origin (which is 0,0).
    """
    return np.sqrt(x**2 + y**2)

def ice_mask(closest_x, closest_y, z, thresh):
    """
    Actually perform the ice masking. If the coordinate is within a certain radiaus and is NaN, assume it is ice (fill in the hole at the North Pole
    from the satellite data), and if not return a 1 if it is sufficiently icy (75% ice) and a 0 if it is not!
    Inputs:
    - closest_x, closest_y, the closest ice coordinate to the trajectory coordinate.
    - z: the corresponding ice coordinate
    Outputs:
    - 1 if attributed to sea-ice, 0 if not sea-ice.
    """
    near_pole = ice_interpolation(closest_x, closest_y) < 325052.227135
    pole_hole = near_pole & np.isnan(z)
    return (pole_hole | (z >= thresh)).astype(int)

"""
Land masking functions
"""

def land_mask(lat, lon):
    """
    Check if a coordinate is land or not using global-land-mask. Global-land-mask cannot be vectorized.
    Inputs:
    - lat, lon: the latitude and longitude coordinate of the point in question
    Outputs:
    - 1 if land and 0 if ocean.
    """
    mask = globe.is_land(lat, lon)
    return mask.astype(int)

"""
Boundary layer masking function
"""

def bl_mask(alt, bl_height):
    """
    Mask to see if the air parcel is above or below the boundary layer.
    Inputs:
    - alt, the simulated altitude of the air parcel
    - bl_height: the simulated altitude of the boundary layer height
    Returns:
    - mask of same dimensions as inputs with 1 where alt > bl_height and 0 otherwise.
    """
    mask = (alt  > bl_height)
    return mask

In [ ]:
#slightly slow but not too slow (~20 seconds)

closest_x = xr.apply_ufunc(
    closest_icecoord,
    ens['x_stereo'],                        
    kwargs={"search_array": ice_dum.x.values},
    vectorize=True,                
    dask="parallelized",           
    output_dtypes=[ice_dum.x.values.dtype],
)

closest_y = xr.apply_ufunc(
    closest_icecoord,
    ens['y_stereo'],
    kwargs={"search_array": ice_dum.y.values},
    vectorize=True,
    dask='parallelized',
    output_dtypes=[ice_dum.y.values.dtype],
)

ens['closest_x'] = closest_x
ens['closest_y'] = closest_y

In [57]:
# selecting the right ice coord for that trajectory location and time
z = ice['z'].sel(x=ens['closest_x'], y=ens['closest_y'])
ens['z'] = z
ens = ens.drop_vars(['x','y'])

In [ ]:
# applying the ice mask, quite slow ~ 2 mins
ens['icemask'] = ice_mask(ens['closest_x'], ens['closest_y'], ens['z'], 75)

In [64]:
# applying the land-ocean mask
landmask = xr.apply_ufunc(
    land_mask,
    ens['lat'],
    ens['lon'],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[int],
)

In [ ]:
# applying the boundary layer mask (should happen last, if it is above the boundary layer we don't care what surface type it is over)

# Original Code From my Masters Thesis

In [ ]:
# Original functions from thesis:

# This is the original function, I need to find a way to vectorize it essentially.

def ice_mask(x_array, y_array, time_array):
    
    #the lists are reset each time we call the function:
    
    timelist=[]
    inputlist=[]
    xlist=[] 
    ylist=[]
 
    for element in time_array:
        
        timelist.append(element)
        
        time_input = np.datetime64(element, 'D').astype(str)
        
        inputlist.append(time_input)
    
    for element in x_array:
        
        x_dif = abs(ice.x.values - element)
        
        closest_x = ice.x.values[np.argmin(x_dif)]
        
        xlist.append(closest_x)
        
    for element in y_array:
        
        y_dif = abs(ice.y.values - element)
        
        closest_y = ice.y.values[np.argmin(y_dif)]
        
        ylist.append(closest_y)
        
    df_dum = pd.DataFrame({'time': timelist, 'time_input': inputlist, 'closest_x': xlist, 'closest_y': ylist,
                          'index': range(0,241)})
    
    df_dum['z'] = df_dum['index'].apply(lambda i: 
                               float(ice['z'].sel(x=df_dum['closest_x'].loc[i],
                                                  y=df_dum['closest_y'].loc[i],
                                                  time=df_dum['time_input'].loc[i]).values))
    
    df_dum['ice_mask'] = df_dum['index'].apply(lambda i: 
                                      1 if ice_interpolation(df_dum['closest_x'].loc[i],
                                                             df_dum['closest_y'].loc[i]) < 325052.227135 and pd.isna(row['z'])
                                      else ice_threshold(df_dum['z'].loc[i]))
    
    df = df_dum.drop(columns=['time_input','index'])
    
    return df